In [1]:
from Parts.imports import *
from Parts.data_loader import DatasetLoader, DatasetLoaderV2
from Parts.models import U_NET_VANILLA, U_NET_RESNET, U_NET_RESNET_ATTENTION, U_NET_PLUS_PLUS, TRANS_U_NET
from Parts.training_loop import TrainingLoop, TrainingLoopAdvanced, SaveState, EarlyStopping
from Parts.losses import DiceLoss, Mixed_Dice_Sigmoid

DEBUG = False

In [2]:
pipe = albumentations.Compose([
    albumentations.OneOf([
        albumentations.ElasticTransform(p=1),
        albumentations.GridDistortion(p=1)
    ]),
    albumentations.CLAHE(),
    albumentations.RandomBrightnessContrast()
])

In [3]:
# image_dir = fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\images"
# label_dir = fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\masks"

# class_instance = DatasetLoaderV2(image_dir, label_dir, (256, 256), pipe)
# data_train, data_val = random_split(class_instance, [0.8, 0.2])
# dataset_loader = DataLoader(data_train,
#                             batch_size=8, shuffle=True) # Shuffle = True shuffles every epoch
# validation_loader = DataLoader(data_val, 8, shuffle=False)

In [6]:
image_dir = fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\images"
label_dir = fr"C:\Users\itizs\Downloads\archive(1)\brisc2025\segmentation_task\train\masks"

# All this pain because we don't want the validation data to be augmented (ie go through the pipe we defined for data augmentation)
# only training data needs to do that, so we load all data, shuffle them, and then load them again with shuffled indices
class_instance = DatasetLoaderV2(image_dir, label_dir, (256, 256), pipe)
indices = numpy.arange(len(class_instance))
numpy.random.shuffle(indices)

split = int(0.8 * len(indices))
train_indices, val_indices = indices[:split], indices[split:]

train_dataset = DatasetLoaderV2(image_dir, label_dir, (256, 256), transformations=pipe)
val_dataset = DatasetLoaderV2(image_dir, label_dir, (256, 256), transformations=None)

data_train = Subset(train_dataset, train_indices) # make a subset of data that only contains the indices we selected
data_val = Subset(val_dataset, val_indices)

# well DataLoader is the class that actually triggers the _getitem()_ of DatasetLoaderV2, so until this point data is never loaded
# hence it is kind of efficient, even though it appears we are using DatasetLoaderV2 thrice
dataset_loader = DataLoader(data_train, batch_size=8, shuffle=True) 
validation_loader = DataLoader(data_val, batch_size=8, shuffle=False) # shuffle shuffls every epoch, we don't want that for validation data


In [ ]:
if DEBUG :
    image, mask = next(iter(dataset_loader))
    plt.imshow(mask[0].permute(1, 2, 0))

In [ ]:
model = U_NET_VANILLA()
optm = torch.optim.Adam(model.parameters(),lr=0.0001)
loss = Mixed_Dice_Sigmoid()
# well the dice loss is causing problems - loss isn't stably and reliably reducing, its, hence mix of bce and dice to be used


In [ ]:
trainer = TrainingLoop(model, loss, optm, 10)
trainer.train(dataset_loader)

In [ ]:
early_stopper = EarlyStopping()
saver = SaveState('SimpleUNet', False)

trainier = TrainingLoopAdvanced(model, loss, optm, 10, early_stopper, saver)

In [ ]:
# from torch.utils.data import Subset # <----------- Such convinient thing they have
# subset_class_instance = Subset(class_instance, [0, 1, 2])
# subset_test_data_loader = DataLoader(subset_class_instance, 1)

# val_subset = Subset(class_instance, [3])
# val_subset_loader = DataLoader(val_subset, 1)

In [ ]:
# trainier.train(subset_test_data_loader, val_subset_loader)

In [ ]:
# well let us see class imbalance
empty_count, non_empty = 0, 0

total = len(dataset_loader)
with tqdm.tqdm(total=total) as bar:
    for data, label in dataset_loader:
        if label.cpu().detach().numpy().unsqueeze().sum() == 0:
            empty_count += 1
        else:
            non_empty += 1
        bar.update(1)

In [ ]:
print(f"empty :: {empty_count} :: Non empty :: {non_empty}")